In [1]:
import pandas as pd

experiments = [ 'lka_micron']
base_path = '/data/shared/fsibilla/clean_code/Q1/experiments/{}/full.csv'

In [2]:
results = []

for exp in experiments:
    path = base_path.format(exp)
    try:
        df = pd.read_csv(path, low_memory=False)
    except FileNotFoundError:
        print(f"NOT FOUND: {exp}")
        continue

    # detect psu column
    psu_col = 'psu' if 'psu' in df.columns else None
    if psu_col is None:
        print(f"{exp}: no 'psu' column found. Columns: {list(df.columns)}")
        continue

    # detect adm columns
    adm1_col = 'adm1name' if 'adm1name' in df.columns else None
    adm2_col = 'adm2name' if 'adm2name' in df.columns else None

    total_rows     = len(df)
    n_unique_global = df[psu_col].nunique()
    n_rows_per_psu  = total_rows / n_unique_global if n_unique_global > 0 else None

    # how many PSUs appear in more than one ADM1?
    psu_crosses_adm1 = None
    if adm1_col:
        psu_adm1_counts = df.groupby(psu_col)[adm1_col].nunique()
        psu_crosses_adm1 = (psu_adm1_counts > 1).sum()

    # how many PSUs appear in more than one ADM2?
    psu_crosses_adm2 = None
    if adm2_col:
        psu_adm2_counts = df.groupby(psu_col)[adm2_col].nunique()
        psu_crosses_adm2 = (psu_adm2_counts > 1).sum()

    # are PSUs unique within each ADM1?
    unique_within_adm1 = None
    if adm1_col:
        # count distinct PSUs per ADM1, sum them, compare to global unique
        n_unique_within_adm1 = df.groupby(adm1_col)[psu_col].nunique().sum()
        unique_within_adm1 = (n_unique_within_adm1 == n_unique_global)

    # are PSUs unique within each ADM2?
    unique_within_adm2 = None
    if adm2_col:
        n_unique_within_adm2 = df.groupby(adm2_col)[psu_col].nunique().sum()
        unique_within_adm2 = (n_unique_within_adm2 == n_unique_global)

    results.append({
        'experiment':         exp,
        'total_rows':         total_rows,
        'unique_PSUs_global': n_unique_global,
        'avg_rows_per_PSU':   round(n_rows_per_psu, 1) if n_rows_per_psu else None,
        'PSUs_cross_adm1':    psu_crosses_adm1,
        'PSUs_cross_adm2':    psu_crosses_adm2,
        'unique_within_adm1': unique_within_adm1,
        'unique_within_adm2': unique_within_adm2,
    })

summary = pd.DataFrame(results)
summary

,experiment,total_rows,unique_PSUs_global,avg_rows_per_PSU,PSUs_cross_adm1,PSUs_cross_adm2,unique_within_adm1,unique_within_adm2
0,lka_micron,19911,2411,8.3,0,0,True,True
